In [1]:
import os
import json
import shutil
import cv2
from ultralytics import YOLO

# ================= MODEL =================
model = YOLO("best2.pt")

CLASSES = [
    "Tshirt","jacket","long-dress","long-skirt","midi-dress","midi-skirt",
    "pants","shirt","short","short-dress","short-skirt","sweater"
]

# ================= PATHS =================
train_img_dir = "clothes-classfy-1/train/images"
train_lbl_dir = "clothes-classfy-1/train/labels"

hata_root = "hatalı_data"
hata_img_dir = os.path.join(hata_root, "images")
hata_lbl_dir = os.path.join(hata_root, "labels")
matris_path = os.path.join(hata_root, "matris.json")

os.makedirs(hata_img_dir, exist_ok=True)
os.makedirs(hata_lbl_dir, exist_ok=True)

# ================= JSON YÜKLE / BAŞLAT =================
if os.path.exists(matris_path):
    try:
        with open(matris_path, "r", encoding="utf-8") as f:
            matris = json.load(f)
            if not isinstance(matris, list):
                matris = []
    except:
        matris = []
else:
    matris = []

hata_counter = len(matris)

# ================= TEST EDİLECEK GÖRSELLER =================
test_limit = 1500
image_files = [f for f in os.listdir(train_img_dir)
               if f.lower().endswith((".jpg", ".jpeg", ".png"))]

image_files = image_files[:test_limit]

print(f"Test edilecek görsel sayısı: {len(image_files)}")

# ================= TEST DÖNGÜSÜ =================
for img_file in image_files:

    img_path = os.path.join(train_img_dir, img_file)
    label_name = os.path.splitext(img_file)[0] + ".txt"
    lbl_path = os.path.join(train_lbl_dir, label_name)

    if not os.path.exists(lbl_path):
        continue

    # ---- Gerçek etiket
    with open(lbl_path, "r") as f:
        line = f.readline().strip()
        if not line:
            continue
        parts = line.split()
        true_cls_id = int(parts[0])
        true_cls_name = CLASSES[true_cls_id]

    # ---- Görsel oku
    frame = cv2.imread(img_path)
    if frame is None:
        print(f"⚠ Okunamadı: {img_path}")
        continue

    # ---- Model tahmini
    results = model.predict(frame, imgsz=640, verbose=False)
    boxes = results[0].boxes

    if boxes is None or len(boxes) == 0:
        predicted_class = "Algılanmadı"
        confidence = 0.0
        bbox = None
    else:
        predicted_class = CLASSES[int(boxes.cls[0])]
        confidence = float(boxes.conf[0])
        bbox = boxes.xywhn[0].tolist()

    # ================= HATALI İSE =================
    if predicted_class != true_cls_name:

        hata_counter += 1
        file_id = f"hata_{hata_counter}"

        # Dosyaları kopyala
        shutil.copy(img_path, os.path.join(hata_img_dir, file_id + ".jpg"))
        shutil.copy(lbl_path, os.path.join(hata_lbl_dir, file_id + ".txt"))

        # JSON objesi (SENİN İSTEDİĞİN FORMAT)
        hata_kaydi = {
            "file_id": file_id,
            "predicted_class": predicted_class,
            "true_class": true_cls_name,
            "confidence": confidence,
            "bbox": bbox
        }

        matris.append(hata_kaydi)

        # 🔥 ANLIK KAYDET (esas düzeltme burada)
        with open(matris_path, "w", encoding="utf-8") as f:
            json.dump(matris, f, indent=2, ensure_ascii=False)

        print(f"❌ Hata kaydedildi → {file_id} | {true_cls_name} → {predicted_class}")

print("\n===== TAMAMLANDI =====")
print(f"Toplam hatalı örnek: {len(matris)}")


Test edilecek görsel sayısı: 1404
❌ Hata kaydedildi → hata_1076 | Tshirt → midi-skirt
❌ Hata kaydedildi → hata_1077 | shirt → sweater
❌ Hata kaydedildi → hata_1078 | pants → midi-skirt
❌ Hata kaydedildi → hata_1079 | Tshirt → midi-skirt
❌ Hata kaydedildi → hata_1080 | pants → short-skirt
❌ Hata kaydedildi → hata_1081 | pants → shirt
❌ Hata kaydedildi → hata_1082 | jacket → short-dress
❌ Hata kaydedildi → hata_1083 | pants → long-skirt
❌ Hata kaydedildi → hata_1084 | shirt → sweater
❌ Hata kaydedildi → hata_1085 | shirt → sweater
❌ Hata kaydedildi → hata_1086 | shirt → sweater
❌ Hata kaydedildi → hata_1087 | pants → midi-skirt
❌ Hata kaydedildi → hata_1088 | Tshirt → long-skirt
❌ Hata kaydedildi → hata_1089 | midi-skirt → short
❌ Hata kaydedildi → hata_1090 | pants → long-skirt
❌ Hata kaydedildi → hata_1091 | shirt → sweater
❌ Hata kaydedildi → hata_1092 | pants → short-skirt
❌ Hata kaydedildi → hata_1093 | midi-skirt → short
❌ Hata kaydedildi → hata_1094 | midi-skirt → short
❌ Hata kay

In [2]:
import json
from collections import Counter, defaultdict
import pandas as pd

# ---------------- JSON YÜKLE ----------------
with open("hatalı_data/matris.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# ---------------- ORİJİNAL 12 SINIF ----------------
CLASSES_12 = [
    "Tshirt","jacket","long-dress","long-skirt","midi-dress","midi-skirt",
    "pants","shirt","short","short-dress","short-skirt","sweater"
]

# ---------------- MERGE MAP (KRİTİK KISIM) ----------------
MERGE_MAP = {
    "long-dress": "dress",
    "midi-dress": "dress",
    "short-dress": "dress",

    "long-skirt": "skirt",
    "midi-skirt": "skirt",
    "short-skirt": "skirt",
}

def merge_class(name):
    return MERGE_MAP.get(name, name)

# ---------------- YENİ SINIF SETİ ----------------
CLASSES_MERGED = [
    "Tshirt","jacket","dress","skirt","pants","shirt","short","sweater","Algılanmadı"
]

# ---------------- SAYIMLAR ----------------
true_counter = Counter()
pred_counter = Counter()
matrix = defaultdict(lambda: defaultdict(int))

for item in data:
    true_cls = merge_class(item["true_class"])
    pred_cls = merge_class(item["predicted_class"])

    true_counter[true_cls] += 1
    pred_counter[pred_cls] += 1
    matrix[true_cls][pred_cls] += 1

# ---------------- GERÇEK SINIF DAĞILIMI ----------------
print("\n===== HATALI ÖRNEKLERDE GERÇEK SINIF DAĞILIMI (MERGED) =====")
for cls, count in true_counter.most_common():
    print(f"{cls}: {count}")

# ---------------- MODELİN YANLIŞ TAHMİN ETTİĞİ SINIFLAR ----------------
print("\n===== MODELİN YANLIŞ TAHMİN ETTİĞİ SINIFLAR (MERGED) =====")
for cls, count in pred_counter.most_common():
    print(f"{cls}: {count}")

# ---------------- CONFUSION MATRIX ----------------
df_matrix = pd.DataFrame(0, index=CLASSES_MERGED, columns=CLASSES_MERGED)

for true_cls in matrix:
    for pred_cls in matrix[true_cls]:
        df_matrix.loc[true_cls, pred_cls] = matrix[true_cls][pred_cls]

print("\n===== CONFUSION MATRIX (MERGED TRUE → PREDICTED) =====")
print(df_matrix)

# ---------------- EN ÇOK KARIŞAN SINIFLAR ----------------
print("\n===== EN ÇOK KARIŞAN SINIF ÇİFTLERİ (MERGED) =====")
pairs = []

for true_cls in matrix:
    for pred_cls in matrix[true_cls]:
        if true_cls != pred_cls:
            pairs.append((true_cls, pred_cls, matrix[true_cls][pred_cls]))

pairs.sort(key=lambda x: x[2], reverse=True)

for t, p, c in pairs[:10]:
    print(f"{t} → {p} : {c} kez")



===== HATALI ÖRNEKLERDE GERÇEK SINIF DAĞILIMI (MERGED) =====
shirt: 508
jacket: 486
skirt: 458
pants: 318
dress: 260
Tshirt: 120

===== MODELİN YANLIŞ TAHMİN ETTİĞİ SINIFLAR (MERGED) =====
dress: 486
sweater: 456
pants: 420
skirt: 354
short: 166
jacket: 156
shirt: 88
Tshirt: 24

===== CONFUSION MATRIX (MERGED TRUE → PREDICTED) =====
             Tshirt  jacket  dress  skirt  pants  shirt  short  sweater  \
Tshirt            0       0      0     38     52      0     28        2   
jacket            0       0    480      2      0      0      0        4   
dress             2     154      0     10     20     72      2        0   
skirt            16       0      0      0    290      8    134       10   
pants             6       0      2    292      0      8      0       10   
shirt             0       2      4     12     58      0      2      430   
short             0       0      0      0      0      0      0        0   
sweater           0       0      0      0      0      0      0  